# 02. DPO 파인튜닝 (Qwen2.5-7B-Instruct + QLoRA)

`01` 노트북에서 구간별로 만든 DPO 데이터(`.jsonl`)를 병합하고, Hugging Face TRL의 `DPOTrainer`로 학습합니다.

| 항목 | 값 |
|---|---|
| 기반 모델 | `Qwen/Qwen2.5-7B-Instruct` (4bit NF4, double quant) |
| LoRA | r=16, alpha=32, dropout=0.05, `q/k/v/o_proj` |
| DPO | beta=0.1, lr=5e-6 (cosine, warmup 0.1), 2 epochs |
| 배치 | 2 × grad accum 4, `max_length=1536` |
| 저장 | `/content/drive/MyDrive/qwen25-7b-dpo-fluff-final` |

GPU 런타임(A100 권장)에서 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. DPO 데이터 병합

`/content`에 업로드한 `dpo_fluff_api_generated_*.jsonl` 파일들을 합치고, 빈 값·완전 중복 레코드를 제거해
`final_dpo_dataset.jsonl`(prompt/chosen/rejected만)과 `final_dpo_dataset_full.jsonl`(메타데이터 포함)을 저장합니다.

In [ ]:
import os
import json

UPLOAD_DIR = "/content"

jsonl_files = [
    os.path.join(UPLOAD_DIR, f)
    for f in os.listdir(UPLOAD_DIR)
    if f.endswith(".jsonl")
]

print(f"감지된 .jsonl 파일 수: {len(jsonl_files)}")
for f in jsonl_files:
    print(" -", f)

In [ ]:
per_file_counts = []
all_records = []

for fname in jsonl_files:
    count = 0
    with open(fname, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"[경고] {fname} 줄 {line_num}: JSON 파싱 실패 -> {e}")
                continue
            record["_source_file"] = fname  # 추적용, 최종 저장 시 제외
            all_records.append(record)
            count += 1
    per_file_counts.append({"file": fname, "loaded_records": count})

total_loaded = len(all_records)
print(f"총 로드된 record 수: {total_loaded}")

In [ ]:
REQUIRED_KEYS = ["prompt", "chosen", "rejected"]

def is_empty(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False

valid_records = []
removed_empty = 0

for record in all_records:
    if any(is_empty(record.get(key)) for key in REQUIRED_KEYS):
        removed_empty += 1
        continue
    valid_records.append(record)

print(f"빈 값으로 제거된 record 수: {removed_empty}")
print(f"검증 통과 record 수: {len(valid_records)}")

In [ ]:
from collections import OrderedDict

seen = OrderedDict()
deduped_records = []
removed_duplicates = 0

for record in valid_records:
    key = (
        record["prompt"].strip(),
        record["chosen"].strip(),
        record["rejected"].strip(),
    )
    if key in seen:
        removed_duplicates += 1
        continue
    seen[key] = True
    deduped_records.append(record)

print(f"중복으로 제거된 record 수: {removed_duplicates}")
print(f"최종 record 수: {len(deduped_records)}")

In [ ]:
final_path = "final_dpo_dataset.jsonl"
with open(final_path, "w", encoding="utf-8") as f:
    for record in deduped_records:
        clean = {
            "prompt": record["prompt"],
            "chosen": record["chosen"],
            "rejected": record["rejected"],
        }
        f.write(json.dumps(clean, ensure_ascii=False) + "\n")

full_path = "final_dpo_dataset_full.jsonl"
with open(full_path, "w", encoding="utf-8") as f:
    for record in deduped_records:
        record_to_save = {k: v for k, v in record.items() if k != "_source_file"}
        f.write(json.dumps(record_to_save, ensure_ascii=False) + "\n")

print(f"{final_path} 저장 완료 ({len(deduped_records)}개)")
print(f"{full_path} 저장 완료 ({len(deduped_records)}개)")

In [ ]:
import pandas as pd

summary_rows = [
    {"file": "TOTAL_LOADED", "loaded_records": total_loaded},
    {"file": "REMOVED_EMPTY", "loaded_records": removed_empty},
    {"file": "REMOVED_DUPLICATES", "loaded_records": removed_duplicates},
    {"file": "FINAL_COUNT", "loaded_records": len(deduped_records)},
]

report_df = pd.DataFrame(per_file_counts + summary_rows)
report_path = "merge_report.csv"
report_df.to_csv(report_path, index=False, encoding="utf-8-sig")

report_df

## 2. 학습 준비

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

In [ ]:
import os

UPLOAD_DIR = "/content"
jsonl_files = [
    os.path.join(UPLOAD_DIR, f)
    for f in os.listdir(UPLOAD_DIR)
    if f.endswith(".jsonl")
]
print(f"감지된 .jsonl 파일: {jsonl_files}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="final_dpo_dataset.jsonl", split="train")

def apply_template(example):
    messages = [{"role": "user", "content": example["prompt"]}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dataset = dataset.map(apply_template)

# 변환 확인
print(dataset[0]["prompt"])

### 토큰 길이 확인

`max_length` 설정을 위해 chosen/rejected 및 prompt+response 길이 분포를 확인합니다.

In [ ]:
lengths_c = [len(tokenizer(ex["chosen"])["input_ids"]) for ex in dataset]
lengths_r = [len(tokenizer(ex["rejected"])["input_ids"]) for ex in dataset]

print(f"chosen 평균: {sum(lengths_c)/len(lengths_c):.0f}, 최대: {max(lengths_c)}")
print(f"rejected 평균: {sum(lengths_r)/len(lengths_r):.0f}, 최대: {max(lengths_r)}")

In [ ]:
#튜닝이전테스트
combined_lengths = []
for ex in dataset:
    prompt_len = len(tokenizer(ex["prompt"])["input_ids"])
    chosen_len = len(tokenizer(ex["chosen"])["input_ids"])
    rejected_len = len(tokenizer(ex["rejected"])["input_ids"])
    combined_lengths.append(prompt_len + max(chosen_len, rejected_len))

print(f"prompt+response 합산 평균: {sum(combined_lengths)/len(combined_lengths):.0f}")
print(f"최대: {max(combined_lengths)}")

In [ ]:
#튜닝이전테스트
import numpy as np

combined_lengths_arr = np.array(combined_lengths)
print(f"50th: {np.percentile(combined_lengths_arr, 50):.0f}")
print(f"75th: {np.percentile(combined_lengths_arr, 75):.0f}")
print(f"90th: {np.percentile(combined_lengths_arr, 90):.0f}")
print(f"95th: {np.percentile(combined_lengths_arr, 95):.0f}")
print(f"99th: {np.percentile(combined_lengths_arr, 99):.0f}")

### train / eval 분리 및 LoRA 설정

In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"train: {len(train_dataset)} / eval: {len(eval_dataset)}")

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

## 3. DPO 학습

In [ ]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    output_dir="/content/drive/MyDrive/qwen25-7b-dpo-fluff",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    bf16=True,
    fp16=False,
    report_to="none",
    max_length=1536,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
final_path = "/content/drive/MyDrive/qwen25-7b-dpo-fluff-final"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print("저장 완료:", final_path)

### 학습 곡선 저장

학습 로그(`trainer.state.log_history`)에서 train/eval loss를 그려 `dpo_training_loss.png`로 저장합니다.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train = [(l["step"], l["loss"]) for l in logs if "loss" in l]
evals = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

plt.figure(figsize=(8, 5))
plt.plot(*zip(*train), marker="o", label="Training Loss")
plt.plot(*zip(*evals), marker="o", label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("DPO Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("dpo_training_loss.png", dpi=150)
plt.show()

## 4. 테스트 및 파인튜닝 전후 비교

In [ ]:
#모델 테스트
def generate_response(prompt_text, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


test_prompt = "I'm elderly 65s, my digital literacy is very low. I got a text message saying my package delivery failed and I need to click a link to pay a small fee to get it redelivered. I don't remember ordering anything. What steps should I take right now to secure my accounts and report this?"

response = generate_response(test_prompt)
print(response)

In [ ]:
#파인튜닝 전후 비교하기
print("---------------- 파인튜닝 전 ----------------")
with trainer.model.disable_adapter():
    inputs = tokenizer(
        tokenizer.apply_chat_template(
            [{"role": "user", "content": test_prompt}],
            tokenize=False, add_generation_prompt=True
        ),
        return_tensors="pt"
    ).to(trainer.model.device)
    with torch.no_grad():
        output = trainer.model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id,
        )
    base_response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(base_response)

print("\n---------------- 파인튜닝 후 (DPO 적용) ----------------")
inputs = tokenizer(
    tokenizer.apply_chat_template(
        [{"role": "user", "content": test_prompt}],
        tokenize=False, add_generation_prompt=True
    ),
    return_tensors="pt"
).to(trainer.model.device)
with torch.no_grad():
    output = trainer.model.generate(
        **inputs, max_new_tokens=512, do_sample=True,
        temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id,
    )
finetuned_response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(finetuned_response)